In [24]:
# Imports
import os
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import biopsykit as bp
import sklearn
from sklearn.preprocessing import MinMaxScaler, StandardScaler, FunctionTransformer
from sklearn.linear_model._cd_fast import ConvergenceWarning

# Regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# Cross-Validation
from sklearn.model_selection import GroupKFold

# Model Evaluation
from biopsykit.classification.model_selection import SklearnPipelinePermuter

In [25]:
# Datapaths
data_path = Path("../../results")
models_path = Path("../../models")
rater = "rater_01"

In [26]:
# Load data
input_data_b_point = pd.read_csv(data_path.joinpath(f"rr-interval/{rater}/train_data_b_point_rr_interval_include_nan_three_best_only.csv"), index_col=[0,1,2,3,4,5]).astype(float)

In [28]:
# Prepare data for training (split features from target and create groups)
X_b_point, y_b_point, groups_b_point, group_keys_b_point = bp.classification.utils.prepare_df_sklearn(data=input_data_b_point, label_col="b_point_sample_reference", subject_col="participant", print_summary=False)
print(f"Input data dtype: {X_b_point.dtype}")
print(f"y_b_point isnan: {np.isnan(y_b_point).any()}")

# Specify model dict
model_dict_b_point = {
    "scaler": {"MinMaxScaler": MinMaxScaler()},
    "clf": {
        "RandomForestRegressor": RandomForestRegressor(),
    },
}

# Fixed hyperparameters (best-performing configuration, cf. Computational_Cost.ipynb / Analyse_Model_Parameters.ipynb)
# instead of searching over a hyperparameter grid.
params_dict_b_point = {
    "MinMaxScaler": None,
    "RandomForestRegressor": {
        "bootstrap": [True],
        "criterion": ["friedman_mse"],
        "max_features": [0.8],
        "n_estimators": [250],
        "ccp_alpha": [0.001],
        "random_state": [0],
    },
}

# Grid search over the single fixed hyperparameter combination above (no actual search performed)
hyper_search_dict = {
    "RandomForestRegressor": {"search_method": "grid"},
}

# Load existing pipeline permuter or create a new one
input_file_path_b_point = models_path.joinpath(f"b-point/rr-interval/{rater}/b_point_include_nan_three_best_only_{rater}.pkl")

pipeline_permuter_b_point = SklearnPipelinePermuter(model_dict_b_point, params_dict_b_point, hyper_search_dict, random_state=0)

# Initialize cross-validation
outer_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=5)
model_name = f"b_point_include_nan_three_best_only_{rater}.pkl"
print(f"Model name: {model_name}")

# Fit and save intermediate results
input_file_path_b_point.parent.mkdir(parents=True, exist_ok=True)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    pipeline_permuter_b_point.fit_and_save_intermediate(X=X_b_point, y=y_b_point, file_path=models_path.joinpath(f"b-point/rr-interval/{rater}/{model_name}"), outer_cv=outer_cv, inner_cv=inner_cv, scoring="neg_mean_absolute_error", groups=groups_b_point)

# Save the fitted pipeline permuter
pipeline_permuter_b_point.to_pickle(models_path.joinpath(f"b-point/rr-interval/{rater}/{model_name}"))
print(f"Generated pickle file: {model_name}")

Input data dtype: float64
y_b_point isnan: False
Model name: b_point_include_nan_three_best_only_rater_01.pkl


Pipeline Combinations:   0%|          | 0/1 [00:00<?, ?it/s]

### Running hyperparameter search for pipeline: (('scaler', 'MinMaxScaler'), ('clf', 'RandomForestRegressor')) with 1 parameter grid(s):
Parameter grid #0 ({'search_method': 'grid'}): {'clf__bootstrap': [True], 'clf__criterion': ['friedman_mse'], 'clf__max_features': [0.8], 'clf__n_estimators': [250], 'clf__ccp_alpha': [0.001], 'clf__random_state': [0]}


Outer CV:   0%|          | 0/5 [00:00<?, ?it/s]

Fitting 5 folds for each of 1 candidates, totalling 5 fits


...

Fitting 5 folds for each of 1 candidates, totalling 5 fits


/Users/abelluc/Code/pepbench/.venv/lib/python3.10/site-packages/biopsykit/classification/model_selection/nested_cv.py:151: UserWarning: Cannot compute confusion matrix for regression tasks.
  warnings.warn("Cannot compute confusion matrix for regression tasks.")


Fitting 5 folds for each of 1 candidates, totalling 5 fits


/Users/abelluc/Code/pepbench/.venv/lib/python3.10/site-packages/biopsykit/classification/model_selection/nested_cv.py:151: UserWarning: Cannot compute confusion matrix for regression tasks.
  warnings.warn("Cannot compute confusion matrix for regression tasks.")




Generated pickle file: b_point_include_nan_three_best_only_rater_01.pkl


In [30]:
# Simple evaluation: MAE fold-wise and sample-wise
from sklearn.metrics import mean_absolute_error

algo_key = ("MinMaxScaler", "RandomForestRegressor")
metric_summary_b_point = pipeline_permuter_b_point.metric_summary().loc[algo_key]

true_labels_folds = metric_summary_b_point["true_labels_folds"]
predicted_labels_folds = metric_summary_b_point["predicted_labels_folds"]

# Fold-wise MAE
mae_per_fold = [mean_absolute_error(t, p) for t, p in zip(true_labels_folds, predicted_labels_folds)]
for i, mae in enumerate(mae_per_fold):
    print(f"Fold {i}: MAE = {mae:.2f} ms")
print(f"Fold-wise MAE: {np.mean(mae_per_fold):.2f} ± {np.std(mae_per_fold):.2f} ms")

# Sample-wise MAE (all folds pooled)
abs_error_sample_wise = np.abs(np.asarray(metric_summary_b_point["true_labels"]) - np.asarray(metric_summary_b_point["predicted_labels"]))
mae_sample_wise = np.mean(abs_error_sample_wise)
std_sample_wise = np.std(abs_error_sample_wise)
print(f"Sample-wise MAE: {mae_sample_wise:.2f} ± {std_sample_wise:.2f} ms")

Fold 0: MAE = 8.96 ms
Fold 1: MAE = 11.11 ms
Fold 2: MAE = 10.60 ms
Fold 3: MAE = 9.77 ms
Fold 4: MAE = 10.80 ms
Fold-wise MAE: 10.25 ± 0.78 ms
Sample-wise MAE: 10.24 ± 12.84 ms


In [31]:
# Statistical comparison against the previously best-performing model
# (MinMax_None_RFR_RR-Interval-Include-Nan, full feature set, reported in the paper)
import pingouin as pg

# Recover the (participant, condition, phase, heartbeat_id_reference) index for every out-of-fold
# prediction, so absolute errors can be matched heartbeat-wise against the previous best model.
test_indices_sample_wise = np.concatenate(metric_summary_b_point["test_indices_folds"])
sample_multiindex = input_data_b_point.index[test_indices_sample_wise].droplevel([0, 5])

new_model_abs_error = pd.Series(abs_error_sample_wise, index=sample_multiindex, name="new_model_abs_error")

previous_best_col = "MinMax_None_RFR_RR-Interval-Include-Nan_abs_error"
previous_best_abs_error = pd.read_csv(
    data_path.joinpath("merged_ml_results_b_point_error_metrics_rr_only.csv"),
    index_col=[0, 1, 2, 3],
)[previous_best_col].rename("previous_best_abs_error")

comparison_df = pd.concat([new_model_abs_error, previous_best_abs_error], axis=1, join="inner")
print(f"Matched heartbeats: {comparison_df.shape[0]} / {new_model_abs_error.shape[0]}")
print(comparison_df.describe())

Matched heartbeats: 11239 / 11239
       new_model_abs_error  previous_best_abs_error
count         11239.000000             11239.000000
mean             10.235837                 8.127195
std              12.841944                12.119868
min               0.000666                 0.000906
25%               2.659998                 1.263296
50%               5.894071                 3.485674
75%              12.337162                10.086356
max             185.778381               189.486654


In [35]:
# Sample-wise paired comparison (non-parametric, cf. established methodology in Compute_Statistical_Significance.ipynb)
normality_results = pg.normality(comparison_df[["new_model_abs_error", "previous_best_abs_error"]])
print("Normality Test Results:\n", normality_results)

wilcoxon_results = pg.wilcoxon(comparison_df["new_model_abs_error"], comparison_df["previous_best_abs_error"])
hedges_g = pg.compute_effsize(
    comparison_df["new_model_abs_error"], comparison_df["previous_best_abs_error"], paired=True, eftype="hedges"
)
wilcoxon_results["hedges_g"] = hedges_g
print("\nWilcoxon signed-rank test (sample-wise):\n", wilcoxon_results)

/Users/abelluc/Code/pepbench/.venv/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 11239.
  res = hypotest_fun_out(*samples, **kwds)


Normality Test Results:
                                 W          pval  normal
new_model_abs_error      0.683062  1.121449e-89   False
previous_best_abs_error  0.629882  2.382830e-93   False

Wilcoxon signed-rank test (sample-wise):
                W-val alternative          p-val       RBC      CLES  hedges_g
Wilcoxon  19699441.0   two-sided  1.790581e-261  0.376237  0.597783  0.168873


In [38]:
# Subject-level aggregation: average per participant first, then re-run the paired test so it is
# not dominated by subjects with many heartbeats.
comparison_df_subject = comparison_df.groupby("participant")[["new_model_abs_error", "previous_best_abs_error"]].mean()

normality_results_subject = pg.normality(comparison_df_subject[["new_model_abs_error", "previous_best_abs_error"]])
print("Normality Test Results (subject-level):\n", normality_results_subject)

wilcoxon_results_subject = pg.wilcoxon(comparison_df_subject["new_model_abs_error"], comparison_df_subject["previous_best_abs_error"])
hedges_g_subject = pg.compute_effsize(
    comparison_df_subject["new_model_abs_error"], comparison_df_subject["previous_best_abs_error"], paired=True, eftype="hedges"
)
wilcoxon_results_subject["hedges_g"] = hedges_g_subject
wilcoxon_results_subject.round(3)

Normality Test Results (subject-level):
                                 W      pval  normal
new_model_abs_error      0.963094  0.225863    True
previous_best_abs_error  0.953632  0.108947    True


,W-val,alternative,p-val,RBC,CLES,hedges_g
Wilcoxon,2.0,two-sided,0.0,0.995,0.673,0.615
